# 🧠 Epileptic Seizure Detection using Deep Learning
**LSTM · 1D-CNN · CNN-LSTM · Random Forest · k-NN · Logistic Regression** — on the UCI / Bonn EEG data set

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/epileptic-seizure-detection/blob/main/notebooks/Epileptic_Seizure_Detection_Colab.ipynb)

This notebook runs the **whole project, step by step**. You do not need to write any code.

### How to use it
1. Look at the **Settings** cell below. **Keep “QUICK_TEST” ticked for your first run** (3–5 minutes, only checks that everything works).
2. Click **Runtime → Run all** (or press `Ctrl + F9`). Do not close the tab.
3. When the notebook asks for the project files, click **Choose Files** and select `epileptic-seizure-detection.zip`
   (not needed if you opened the notebook from your GitHub repository).
4. For the **real training** **untick “QUICK_TEST”** and run all cells again (about 20–30 minutes).
5. The last cell downloads `results_colab.zip` with all tables, pictures and trained models.

### Contents
0. Settings · 1. Get the project files · 2. Check the computer · 3. Look at the data · 4. Prepare the data ·
5. Machine-learning models · 6. Deep-learning models · 7. Training · 8. Machine learning + deep learning combined ·
9. Results on the test data · 10. See some predictions · 11. Honest check (recordings kept together) · 12. Save and download

## 0️⃣ Settings
Only change the values in this cell (tick/untick the boxes, type your GitHub username if you use GitHub).

In [ ]:
#@title Settings { display-mode: "both" }
# In Google Colab the lines below with  #@param  appear as a checkbox / text box - just click or type, no coding needed.

QUICK_TEST = True            #@param {type:"boolean"}
# True  = quick 3-5 minute test (tiny training, results are NOT final)
# False = the REAL training (about 20-30 minutes)

RUN_LEAK_FREE_CHECK = True   #@param {type:"boolean"}
# True  = at the end also repeat everything with the "recording split" (honest check)

GITHUB_USERNAME = "YOUR-GITHUB-USERNAME"   #@param {type:"string"}
REPO_NAME = "epileptic-seizure-detection"  #@param {type:"string"}
# ^ Only needed if you want Colab to download the project from YOUR GitHub repository.
#   If you will upload the project ZIP instead, leave them as they are.

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
OUT_DIR = "results_colab"    # everything this notebook creates is saved in this folder

## 1️⃣ Get the project files
This cell finds the project folder. It tries, in this order:
1. you are already inside the project folder → nothing to do;
2. you typed your **GitHub username** in the Settings box → it downloads the project from your repository;
3. otherwise → a **Choose Files** button appears: select `epileptic-seizure-detection.zip`.

In [ ]:
import os, sys, glob, zipfile, subprocess

def in_project():
    return os.path.isdir("src") and os.path.isdir("scripts")

def find_project_zip():
    """Newest ZIP (any name) that really contains this project."""
    for z in sorted(glob.glob("*.zip") + glob.glob("/content/*.zip"), key=os.path.getmtime, reverse=True):
        try:
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith("src/__init__.py") for n in zf.namelist()):
                    return z
        except zipfile.BadZipFile:
            pass
    return None

if not in_project() and os.path.isdir("../src"):          # notebook opened from the 'notebooks' folder
    os.chdir("..")

if not in_project():
    if "YOUR-GITHUB-USERNAME" not in REPO_URL:             # way 1: clone from GitHub
        print("Downloading the project from GitHub ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir(os.path.basename(REPO_URL).replace(".git", ""))
    else:                                                  # way 2: use the ZIP file
        zip_path = find_project_zip()
        if zip_path is None:
            try:
                from google.colab import files
                print("👉 Click 'Choose Files' and select  epileptic-seizure-detection.zip")
                files.upload()
            except ImportError:
                raise SystemExit("Open this notebook from inside the project folder, or type your GitHub username in the Settings cell.")
            zip_path = find_project_zip()
        if zip_path is None:
            raise SystemExit("No project ZIP found. Run this cell again and select epileptic-seizure-detection.zip")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall("project_unzipped")
        hits = glob.glob("project_unzipped/**/src/__init__.py", recursive=True)
        os.chdir(os.path.dirname(os.path.dirname(hits[0])))

sys.path.insert(0, os.getcwd())
print("✅ Project folder:", os.getcwd())
print("   data file present:", os.path.exists("data/data.csv"))

## 2️⃣ Check the computer
TensorFlow is the library that builds and trains the neural networks. A GPU makes training faster but is **not required**.

*You may see a few pink/red lines about “oneDNN” or “cuda” – they are harmless information messages, not errors.*

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"                    # keep the output quiet
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image
import tensorflow as tf
tf.get_logger().setLevel("ERROR")                           # hide harmless TensorFlow warnings

print("TensorFlow version:", tf.__version__)
print("GPU found ✅" if tf.config.list_physical_devices("GPU") else "No GPU - using the CPU (this is fine, just a bit slower)")
if QUICK_TEST:
    print("\n⚠️  QUICK TEST MODE: tiny training on 20% of the data. The numbers below are NOT final results!")
else:
    print("\n✅ FULL MODE: real training (this will take a while).")

## 3️⃣ Look at the data
* Every row = **178 EEG numbers (1 second of brain signal)** + a label `y`.
* `y = 1` → **seizure**; `y = 2, 3, 4, 5` → **no seizure**.
* The first column is an ID. The part after `X21.` tells us **which recording** a row came from (used for the honest check in step 11).

In [ ]:
from src import config, data, ml, models, hybrid, evaluate as ev
from src import train as T

df = data.load_dataframe()
if QUICK_TEST:
    df = data.quick_subsample(df, 0.2)          # keep 20% of the recordings for the quick test
print("Rows:", len(df), "| columns:", df.shape[1])
df.iloc[:5, :8]

In [ ]:
ev.plot_class_distribution(df, show=True)
ev.plot_sample_eeg(df, data.FEATURE_COLUMNS, show=True)

## 4️⃣ Prepare the data
1. **Binary labels:** seizure = 1, everything else = 0.
2. **Split 7 : 1 : 2** → 70 % training (learning), 10 % validation (choosing settings, stopping in time), 20 % test (final exam).
3. **Normalise** every column to mean 0 / standard deviation 1 (numbers learned from the training data only).

This is the *random split* used in the project report. (Step 11 shows the stricter “recording split”.)

In [ ]:
splits = data.prepare(df, "random", seed=config.SEED)
i = splits.info
print("rows        train / validation / test :", i["rows"]["train"], "/", i["rows"]["val"], "/", i["rows"]["test"])
print("seizures    train / validation / test :", i["seizure_rows"]["train"], "/", i["seizure_rows"]["val"], "/", i["seizure_rows"]["test"])
print("recordings that appear in BOTH train and test:", i["recordings_shared_between_train_and_test"], "of", i["recordings"]["test"], "test recordings")
print("shape of the training data:", splits.X_train.shape, " (rows, 178 EEG values)")

## 5️⃣ Machine-learning models
**Random Forest**, **k-Nearest Neighbours** and **Logistic Regression**. For each one the program tries several settings
(from the report), scores them on the *validation* data and keeps the best. This takes 1–3 minutes.

In [ ]:
ml_results = ml.run_ml_models(splits, quick=QUICK_TEST)

## 6️⃣ Deep-learning models
The three networks of the report. Here is the best one, the **CNN-LSTM**: convolution layers find small wave shapes,
LSTM layers follow how they change over time, dense layers decide. (Trainable parameters: LSTM 197,090 · CNN 8,626 · CNN-LSTM 127,578.)

In [ ]:
cnn_lstm_preview = models.build("CNN-LSTM")
cnn_lstm_preview.summary()

## 7️⃣ Training
Each network sees the training data again and again (one pass = one *epoch*). After each epoch it is checked on the validation data.
**Early stopping** ends the training when the validation loss stops improving, and goes back to the best epoch.
Watch `val_accuracy` going up. (The LSTM may stay at 0.80 for a few epochs before it “wakes up” – that is normal.)

In [ ]:
EPOCHS = 2 if QUICK_TEST else config.EPOCHS
PATIENCE = 3 if QUICK_TEST else config.EARLY_STOP_PATIENCE
dl, dl_info = {}, {}
OUT = os.path.join(OUT_DIR, "random_split")
os.makedirs(os.path.join(OUT, "figures"), exist_ok=True)
os.makedirs(os.path.join(OUT, "models"), exist_ok=True)
os.makedirs(os.path.join(OUT, "tables"), exist_ok=True)

In [ ]:
# ---- LSTM ----
dl["LSTM"], dl_info["LSTM"] = T.train_dl_model("LSTM", splits, epochs=EPOCHS, patience=PATIENCE, verbose=2)
print("trained epochs:", dl_info["LSTM"]["epochs_run"], "| best epoch:", dl_info["LSTM"]["best_epoch"],
      "| time: %.0f s" % dl_info["LSTM"]["train_seconds"])
ev.plot_history(dl_info["LSTM"]["history"], "LSTM", show=True)

In [ ]:
# ---- 1D-CNN ----
dl["CNN"], dl_info["CNN"] = T.train_dl_model("CNN", splits, epochs=EPOCHS, patience=PATIENCE, verbose=2)
print("trained epochs:", dl_info["CNN"]["epochs_run"], "| best epoch:", dl_info["CNN"]["best_epoch"],
      "| time: %.0f s" % dl_info["CNN"]["train_seconds"])
ev.plot_history(dl_info["CNN"]["history"], "CNN", show=True)

In [ ]:
# ---- CNN-LSTM (the longest one) ----
dl["CNN-LSTM"], dl_info["CNN-LSTM"] = T.train_dl_model("CNN-LSTM", splits, epochs=EPOCHS, patience=PATIENCE, verbose=2)
print("trained epochs:", dl_info["CNN-LSTM"]["epochs_run"], "| best epoch:", dl_info["CNN-LSTM"]["best_epoch"],
      "| time: %.0f s" % dl_info["CNN-LSTM"]["train_seconds"])
ev.plot_history(dl_info["CNN-LSTM"]["history"], "CNN-LSTM", show=True)

## 8️⃣ Machine learning + deep learning combined
The abstract of the report wants to *combine* machine learning and deep learning. Two simple ways:
* **CNN-LSTM + RF:** the CNN-LSTM turns every EEG second into 32 learned numbers, a Random Forest decides from them.
* **Ensemble:** LSTM, CNN, CNN-LSTM and Random Forest vote (average of their seizure probabilities).

In [ ]:
prob_test = {name: r["prob_test"] for name, r in ml_results.items()}
for name, model in dl.items():
    prob_test[name] = T.predict_prob(name, model, splits.X_test)

hy = hybrid.cnn_lstm_plus_rf(dl["CNN-LSTM"], splits, quick=QUICK_TEST)
prob_test["CNN-LSTM + RF"] = hy["prob_test"]
members = ["LSTM", "CNN", "CNN-LSTM", "Random Forest"]
prob_test["Ensemble (all models)"] = hybrid.soft_vote([prob_test[n] for n in members])
print("models ready:", list(prob_test))

## 9️⃣ Results on the test data (data the models have never seen)
All numbers are for the **seizure class**:
* **Precision** – when the model says “seizure”, how often is it right?
* **Recall (sensitivity)** – how many of the real seizures were found?
* **Specificity** – how many normal signals were correctly left alone?
* **F1** – balance of precision and recall.
* **95 % CI** – the range in which the true value very likely lies (we re-draw whole recordings 2,000 times).
  If the ranges of two models overlap a lot, you cannot honestly say that one is better than the other.

In [ ]:
metrics = {name: ev.binary_metrics(splits.y_test, p) for name, p in prob_test.items()}
table = ev.results_table(metrics)
table = ev.add_confidence_intervals(table, splits.y_test, prob_test, splits.groups_test)
table["Trainable params"] = [dl_info[n]["params_trainable"] if n in dl_info else "" for n in table["Model"]]
table["Train time (s)"] = [round(dl_info[n]["train_seconds"]) if n in dl_info else "" for n in table["Model"]]
display(table[["Model", "Accuracy %", "Precision %", "Recall % (sensitivity)", "Specificity %", "F1", "ROC-AUC",
               "Accuracy 95% CI", "F1 95% CI", "Trainable params", "Train time (s)"]].round(3))

In [ ]:
fig_dir = os.path.join(OUT, "figures")
ev.plot_comparison(table, "Test-set results (random split)", os.path.join(fig_dir, "comparison.png"), show=True)
ev.plot_confusion_grid(metrics, "Confusion matrices - test set (random split)", os.path.join(fig_dir, "confusion_matrices.png"), show=True)
ev.plot_roc(splits.y_test, prob_test, "ROC curves - test set (random split)", os.path.join(fig_dir, "roc_curves.png"), show=True)
ev.plot_history_grid({n: i["history"] for n, i in dl_info.items()}, "Learning curves (random split)",
                     os.path.join(fig_dir, "training_curves.png"), show=True)

**How to read a confusion matrix:** the top-left box = normal signals correctly called normal; the bottom-right box = seizures correctly found;
the bottom-left box = **missed seizures** (the dangerous mistake); the top-right box = false alarms.

## 🔟 See some predictions
Three real seizure seconds and three normal seconds from the **test** set, and what the CNN-LSTM says.

In [ ]:
rng = np.random.default_rng(0)
idx = np.concatenate([rng.choice(np.where(splits.y_test == 1)[0], 3, replace=False),
                      rng.choice(np.where(splits.y_test == 0)[0], 3, replace=False)])
fig, axes = plt.subplots(2, 3, figsize=(13, 5.5))
for ax, k in zip(axes.ravel(), idx):
    p = prob_test["CNN-LSTM"][k]
    truth, pred = int(splits.y_test[k]), int(p >= 0.5)
    ax.plot(splits.X_test[k], lw=1, color="#d62728" if truth else "#1f77b4")
    ax.set_title(f"true: {'SEIZURE' if truth else 'normal'} | model: {'SEIZURE' if pred else 'normal'} "
                 f"(p={p:.2f}) -> {'correct' if truth == pred else 'WRONG'}", fontsize=9)
    ax.set_xlabel("sample number (normalised EEG)")
plt.tight_layout(); plt.show()

## 1️⃣1️⃣ Honest check – recordings kept together
Every EEG second comes from a longer recording, and neighbouring seconds continue each other.
With the *random split* above, seconds of the **same recording** can be in training **and** test, which can make the test easier.
Here we repeat everything so that **all seconds of a recording stay together**. Now the test contains only recordings the model has never seen.
This is the fair estimate of how the model behaves on a new recording. Compare it with step 9: if the numbers are close, the model really works on unseen recordings.

In [ ]:
if RUN_LEAK_FREE_CHECK:
    cmd = f"python scripts/run_all.py --split recording --out {OUT_DIR}" + (" --quick" if QUICK_TEST else "")
    print("running:", cmd, "\n")
    !{cmd}
else:
    print("Skipped (RUN_LEAK_FREE_CHECK = False)")

In [ ]:
rec_csv = os.path.join(OUT_DIR, "recording_split", "tables", "results_test.csv")
if RUN_LEAK_FREE_CHECK and os.path.exists(rec_csv):
    rec = pd.read_csv(rec_csv)
    display(rec[["Model", "Accuracy %", "Precision %", "Recall % (sensitivity)", "Specificity %", "F1", "ROC-AUC",
                 "Accuracy 95% CI", "F1 95% CI"]].round(3))
    display(Image(os.path.join(OUT_DIR, "recording_split", "figures", "comparison.png")))

## 1️⃣2️⃣ Save and download everything

In [ ]:
# save the random-split results (tables + trained models) next to the pictures that were saved above
table.to_csv(os.path.join(OUT, "tables", "results_test.csv"), index=False)
for name, model in dl.items():
    T.save_model(model, os.path.join(OUT, "models", ev.slug(name) + ".keras"))
data.save_scaler(splits.scaler, os.path.join(OUT, "models", "scaler.json"))
ev.save_json(splits.info, os.path.join(OUT, "tables", "split_summary.json"))

import shutil
shutil.make_archive("results_colab", "zip", OUT_DIR)
print("Created results_colab.zip  (%.1f MB)" % (os.path.getsize("results_colab.zip") / 1e6))
try:
    from google.colab import files
    files.download("results_colab.zip")          # starts the download in your browser
except ImportError:
    print("(not running in Colab - the zip file is in the project folder)")
if QUICK_TEST:
    print("\n⚠️  This was a QUICK TEST. Untick QUICK_TEST in the Settings cell and run all cells again for the real results.")

## ✅ Finished! What next?
* Compare the table of step 9 with the report (and read `docs/4_NOTES_FOR_YOUR_REPORT.md` for important corrections).
* Put the project on GitHub: `docs/2_PUSH_TO_GITHUB.md`.
* Run the detector on a Raspberry Pi: `raspberry_pi/README.md`.
* Dataset credit: Andrzejak et al., *Phys. Rev. E* 64, 061907 (2001) · Wu & Fokoué, UCI Machine Learning Repository (2017).